# YieldGuard — does self-training improve WaferCNN?

WM-811K has **811,457 wafer maps, but only ~173K are labelled**. The idea under test:
let the trained CNN label the unlabelled maps ("pseudo-labels"), add the confident ones
to the training set, and retrain.

The known risk is that the model learns its own mistakes. So this notebook is built to
give a **measured** answer rather than a hopeful one:

1. **A clean test set.** Validation is split in half. One half picks checkpoints and sets
   thresholds; the other half is used exactly once, at the end.
2. **A clean teacher.** Pseudo-labels come from a baseline trained here, which never saw
   the test half.
3. **Same recipe for both arms.** Baseline (human labels only) and self-trained
   (human + pseudo) differ only in the extra data.
4. **Two seeds per arm**, so a real gain can be told apart from run-to-run noise.
5. **Quality filters** learned from inspecting the pseudo-labels: per-class thresholds,
   agreement across all 8 rotations/mirrors, and filters for three test-artifact patterns
   that the model labels with near-100% confidence.

**Runtime → Change runtime type → T4 GPU.** About 45–50 minutes end to end (4 training runs).

In [ ]:
#@title 0. Settings — every knob is here
N_SAMPLE    = 100_000   # unlabelled maps to score (None = all ~638K; ~6x slower to score)
INCLUDE     = ["Center", "Near-full", "Scratch"]   # classes allowed to receive pseudo-labels
SEEDS       = [0, 1]    # seeds per arm; [0] halves the time but gives no noise estimate
EPOCHS      = 40        # same recipe as the shipped model
TARGET_PREC = 0.95      # kept pseudo-labels must be this precise on the selection half
CAP         = 1.0       # pseudo-labels per class <= CAP x that class's human count
PSEUDO_W    = 0.5       # weight of a pseudo-label in the loss (human = 1.0)
BANDED_MAX  = 20        # cap on the "fixed passing bands" Near-full template
STRIPE      = 0.25      # stripe-score cut-off for the striping artifact

# Classes NOT in INCLUDE, and why (from inspecting pseudo-labels by eye):
#   None       — already plentiful
#   Edge-Ring  — 8,228 human labels, F1 0.98; 16% of its pseudo-labels flagged as artifacts
#   Random     — ~25% wrong even after artifact filtering: the model's "catch-all" class
#   Local, Edge-Loc, Donut — ~4-6 of 24 wrong by eye

In [ ]:
#@title 1. Install + Kaggle credentials
!pip -q install kagglehub

import os
from getpass import getpass
if not os.environ.get("KAGGLE_KEY"):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
        print("using Colab Secrets")
    except Exception:
        os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
        os.environ["KAGGLE_KEY"]      = getpass("Kaggle key: ").strip()
print("credentials set for:", os.environ.get("KAGGLE_USERNAME"))

In [ ]:
#@title 2. Data — labelled set (same split as train_simple) + an unlabelled sample
import numpy as np, sys, pickle, warnings
warnings.filterwarnings("ignore")
from PIL import Image
from sklearn.model_selection import train_test_split

CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Local","Random","Scratch","Near-full","None"]
NC = 9
CACHE, UCACHE = "wm811k_64.npz", f"wm811k_unlabelled_{N_SAMPLE}.pkl"

def resize64(wm):
    # NEAREST: pixels are categories (0 not tested, 1 pass, 2 fail), not intensities
    return np.array(Image.fromarray(wm.astype(np.uint8), "L").resize((64, 64), Image.NEAREST),
                    dtype=np.uint8)

if not (os.path.exists(CACHE) and os.path.exists(UCACHE)):
    import kagglehub, pandas as pd
    pkl = os.path.join(kagglehub.dataset_download("qingyi/wm811k-wafer-map"), "LSWMD.pkl")
    # LSWMD.pkl was pickled by pandas <0.20 on Python 2 — needs these shims to load
    import pandas.core.indexes as _ci
    sys.modules.setdefault("pandas.indexes", _ci)
    sys.modules.setdefault("pandas.indexes.base", _ci.base)
    sys.modules.setdefault("pandas.indexes.numeric", getattr(_ci, "numeric", _ci.base))
    # pickle is WM-811K's own distribution format — the only way the dataset ships
    with open(pkl, "rb") as f:
        df = pickle.load(f, encoding="latin1")

    fix = {"none":"None","center":"Center","donut":"Donut","edge-loc":"Edge-Loc",
           "edge-ring":"Edge-Ring","loc":"Local","local":"Local","random":"Random",
           "scratch":"Scratch","near-full":"Near-full","nearfull":"Near-full"}
    def label_of(ft):
        if isinstance(ft, np.ndarray): ft = ft.tolist()
        if not ft or not ft[0]: return None
        s = str(ft[0][0] if isinstance(ft[0], list) else ft[0]).strip()
        return fix.get(s.lower(), s)
    df["label"] = [label_of(v) for v in df["failureType"]]

    if not os.path.exists(CACHE):        # identical to train_simple.ipynb, so the split matches
        lab = df[df["label"].notna()]
        none_m = lab["label"] == "None"
        use = pd.concat([lab[~none_m], lab[none_m].sample(frac=0.25, random_state=42)],
                        ignore_index=True)
        X, y = [], []
        for wm, lb in zip(use["waferMap"], use["label"]):
            if not isinstance(wm, np.ndarray) or wm.size == 0: continue
            X.append(resize64(wm)); y.append(CLASSES.index(lb))
        X = np.stack(X)[:, None]; y = np.array(y, dtype=np.int64)
        Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
        np.savez_compressed(CACHE, X_train=Xtr, y_train=ytr, X_val=Xva, y_val=yva)

    unl = df[df["label"].isna()]
    N_UNL = len(unl)
    if N_SAMPLE and N_SAMPLE < N_UNL:
        unl = unl.sample(n=N_SAMPLE, random_state=42)
    # keep ORIGINAL maps: the artifact filters need the un-resized die grid
    maps = [w for w in unl["waferMap"] if isinstance(w, np.ndarray) and w.ndim == 2 and w.size]
    with open(UCACHE, "wb") as f:
        pickle.dump({"maps": maps, "n_unlabelled": N_UNL}, f)
    del df, unl

d = np.load(CACHE)
X_train = d["X_train"].astype(np.float32) / 2.0     # {0,1,2} -> {0, 0.5, 1}
X_val   = d["X_val"].astype(np.float32)   / 2.0
y_train, y_val = d["y_train"].astype(np.int64), d["y_val"].astype(np.int64)
with open(UCACHE, "rb") as f:     # written by this notebook above, not downloaded
    u = pickle.load(f)
maps, N_UNL = u["maps"], u["n_unlabelled"]

# Split human-labelled val: one half selects checkpoints + sets thresholds, the other is TEST.
sel_i, test_i = train_test_split(np.arange(len(y_val)), test_size=0.5,
                                 stratify=y_val, random_state=42)
Xs, ys, Xt, yt = X_val[sel_i], y_val[sel_i], X_val[test_i], y_val[test_i]

print(f"train {len(y_train):,}   selection half {len(ys):,}   test half {len(yt):,}")
print(f"unlabelled maps {N_UNL:,}   sampled {len(maps):,}")

In [ ]:
#@title 3. Model + training helpers — WaferCNN must match src/models/vision/model.py
import copy, time, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("!! no GPU — Runtime > Change runtime type > T4 GPU, or this takes many hours")

class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                          nn.BatchNorm2d(cout))
    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(h)) + self.shortcut(x))

class WaferCNN(nn.Module):
    def __init__(self, nc=NC, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.layer1 = ResBlock(16,  32,  stride=2)
        self.layer2 = ResBlock(32,  64,  stride=2)
        self.layer3 = ResBlock(64,  128, stride=2)
        self.layer4 = ResBlock(128, 128, stride=2)
        self.gap, self.dropout, self.fc = nn.AdaptiveAvgPool2d(1), nn.Dropout(dropout), nn.Linear(128, nc)
    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        return self.fc(self.dropout(self.gap(x).flatten(1)))

assert sum(p.numel() for p in WaferCNN().parameters()) == 615801, "architecture drifted"

VIEWS = [(k, f) for k in range(4) for f in (False, True)]   # 4 rotations x 2 mirrors

@torch.no_grad()
def tta(m, X, bs=2048):
    """Averaged softmax over the 8 views, plus each view's own prediction (for agreement)."""
    m.eval(); T = torch.from_numpy(X)
    probs = torch.zeros(len(X), NC); votes = torch.zeros(len(X), 8, dtype=torch.long)
    for vi, (k, flip) in enumerate(VIEWS):
        V = torch.rot90(T, k, [-2, -1])
        if flip: V = V.flip(-1)
        for i in range(0, len(V), bs):
            p = F.softmax(m(V[i:i+bs].to(DEVICE)), 1).cpu()
            probs[i:i+bs] += p; votes[i:i+bs, vi] = p.argmax(1)
    return (probs / 8).numpy(), votes.numpy()

def macro(y, p):
    # labels=range(NC): a class the model never predicts scores 0, not dropped from the mean
    return f1_score(y, p, average="macro", labels=range(NC), zero_division=0)

class AugData(Dataset):
    """Dihedral augmentation (labels survive flips/rotations) plus a per-sample loss weight."""
    def __init__(self, X, y, sw):
        self.X, self.y = torch.from_numpy(X), torch.from_numpy(y)
        self.sw = torch.from_numpy(sw.astype(np.float32))
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.X[i]
        if torch.rand(1).item() > .5: x = x.flip(-1)
        if torch.rand(1).item() > .5: x = x.flip(-2)
        k = int(torch.randint(0, 4, (1,)).item())
        if k: x = torch.rot90(x, k, [-2, -1])
        return x, self.y[i], self.sw[i]

def train(X, y, sw, seed, tag):
    """The shipped recipe: sqrt inverse-frequency weights, Adam 1e-3 wd 1e-4, cosine LR,
    batch 128, checkpoint = best macro-F1 on the SELECTION half (never the test half)."""
    torch.manual_seed(seed); np.random.seed(seed)
    counts = np.clip(np.bincount(y, minlength=NC), 1, None).astype(float)
    w = (1.0 / counts) ** 0.5
    w = torch.tensor(w / w.sum() * NC, dtype=torch.float, device=DEVICE)
    tl = DataLoader(AugData(X, y, sw), batch_size=128, shuffle=True,
                    num_workers=2, pin_memory=True)
    m = WaferCNN().to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w, reduction="none")
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    best, best_state, t0 = -1.0, None, time.time()
    for ep in range(1, EPOCHS + 1):
        m.train()
        for xb, yb, sb in tl:
            xb, yb, sb = (t.to(DEVICE, non_blocking=True) for t in (xb, yb, sb))
            opt.zero_grad()
            # weighted mean — identical to the shipped loss when every sample weight is 1
            loss = (crit(m(xb), yb) * sb).sum() / (w[yb] * sb).sum()
            loss.backward(); opt.step()
        sched.step()
        m.eval()
        with torch.no_grad():
            p = torch.cat([m(torch.from_numpy(Xs[i:i+1024]).to(DEVICE)).argmax(1).cpu()
                           for i in range(0, len(Xs), 1024)]).numpy()
        f = macro(ys, p)
        if f > best: best, best_state = f, copy.deepcopy(m.state_dict())
        if ep % 10 == 0:
            print(f"  [{tag}] epoch {ep}/{EPOCHS}  selection macro-F1 {f:.4f}  best {best:.4f}"
                  f"  ({time.time()-t0:.0f}s)")
    m.load_state_dict(best_state)
    return m

def score(m):
    p = tta(m, Xt)[0].argmax(1)
    return macro(yt, p), f1_score(yt, p, average=None, labels=range(NC), zero_division=0)

print("helpers ready")

In [ ]:
#@title 4. Artifact filters — three test-artifact patterns found in the unlabelled data
# All three sit outside WM-811K's nine classes, and the CNN labels them with ~100% confidence.

def purity(m):
    """Mean over rows of |2*fail_rate - 1|: 1 = each row all-fail or all-pass, 0 = half/half."""
    n = (m > 0).sum(1); k = n >= 5
    return np.abs(2 * (m == 2).sum(1)[k] / n[k] - 1).mean() if k.any() else 0.0

def stripe_score(wm):
    """Row-pair STRIPING: rows uniform but columns mixed (or the reverse) -> ~1.
    Scatter (mixed both ways) and near-full/clean (uniform both ways) -> ~0."""
    return abs(purity(wm) - purity(wm.T))

def gappy(wm, frac=0.15):
    """INTERLEAVED untested rows/columns inside the wafer's span."""
    out = 0.0
    for m in (wm, wm.T):
        rows = np.where((m > 0).any(1))[0]
        if len(rows) < 6: continue
        out = max(out, ((m[rows.min(): rows.max() + 1] > 0).sum(1) == 0).mean())
    return out > frac

def banded(a):
    """FIXED PASSING BANDS: the whole wafer failed except a few fully-passing rows."""
    n = (a > 0).sum(1); k = n >= 10
    return bool((a == 2).sum() / max((a > 0).sum(), 1) >= 0.8
                and ((a == 1).sum(1)[k] / n[k] >= 0.9).sum() >= 2)

def artifact(wm):
    return stripe_score(wm) >= STRIPE or gappy(wm)

# self-check on synthetic 40x40 wafers — fails loudly if a filter is broken
_Y, _X = np.mgrid[0:40, 0:40]; _disc = np.hypot(_Y - 19.5, _X - 19.5) <= 19.5
_rng = np.random.default_rng(0)
def _wafer(fill):
    a = np.zeros((40, 40), int); a[_disc] = 1; a[_disc & fill] = 2; return a
_stripes = _wafer(((_Y // 2) % 2 == 0))                         # real pattern: 2 fail, 2 pass
_scatter = _wafer(_rng.random((40, 40)) < 0.5)
_gaps    = _wafer(_rng.random((40, 40)) < 0.3); _gaps[1::2] = 0
_bands   = _wafer(np.ones((40, 40), bool)); _bands[[8, 20, 32]] = np.where(_disc[[8, 20, 32]], 1, 0)
assert artifact(_stripes) and artifact(_gaps), "artifact filter misses a known artifact"
assert not artifact(_scatter), "artifact filter flags plain scatter"
assert banded(_bands) and not banded(_scatter), "banded filter broken"
print("artifact filters OK")

In [ ]:
#@title 5. Baseline arm — human labels only  (~8 min per seed)
results, teacher = {"baseline": [], "self-trained": []}, None
for s in SEEDS:
    m = train(X_train, y_train, np.ones(len(y_train)), s, f"baseline seed {s}")
    results["baseline"].append(score(m))
    print(f"baseline      seed {s}: TEST macro-F1 {results['baseline'][-1][0]:.4f}")
    if teacher is None:
        teacher = m     # the teacher has never seen the test half
torch.save(teacher.state_dict(), "teacher.pt")

In [ ]:
#@title 6. Pseudo-label the unlabelled sample — thresholds from the selection half only
pv, vv = tta(teacher, Xs)
pred_v, conf_v = pv.argmax(1), pv.max(1)
agree_v = (vv == pred_v[:, None]).all(1)
grid, thr = np.round(np.arange(0.50, 1.0001, 0.01), 2), {}
for c in [CLASSES.index(n) for n in INCLUDE]:
    cand = (pred_v == c) & agree_v
    thr[c] = next((t for t in grid if (cand & (conf_v >= t)).sum() >= 5
                   and (ys[cand & (conf_v >= t)] == c).mean() >= TARGET_PREC), None)

U, usrc = [], []
for j, wm in enumerate(maps):
    a = resize64(wm)
    if a.max() <= 2 and 0.20 <= (a > 0).mean() <= 0.95:     # plausible wafer
        U.append(a); usrc.append(j)
U = np.stack(U)
pu, vu = np.zeros((len(U), NC), np.float32), np.zeros((len(U), 8), np.int64)
for i in range(0, len(U), 20_000):
    pu[i:i+20_000], vu[i:i+20_000] = tta(teacher, U[i:i+20_000, None].astype(np.float32) / 2.0)
pr_u, cf_u = pu.argmax(1), pu.max(1)
ag_u = (vu == pr_u[:, None]).all(1)

human_n, rng = np.bincount(y_train, minlength=NC), np.random.default_rng(0)
picked = {}
print(f"scored {len(U):,} plausible maps   all-8-views agree {ag_u.mean():.1%}\n")
print(f"{'class':<11}{'thresh':>7}{'pass':>7}{'artifact':>10}{'banded':>8}{'added':>7}{'human':>8}")
for c, t in thr.items():
    if t is None:
        print(f"{CLASSES[c]:<11}{'none':>7}   threshold unreachable at {TARGET_PREC:.0%}, skipped")
        continue
    idx = np.where((pr_u == c) & ag_u & (cf_u >= t))[0]
    art = np.array([artifact(maps[usrc[i]]) for i in idx], bool)
    keep = idx[~art]
    nb = 0
    if CLASSES[c] == "Near-full" and len(keep):
        b = np.array([banded(U[i]) for i in keep], bool)
        nb = b.sum()
        keep = np.concatenate([keep[~b], rng.permutation(keep[b])[:BANDED_MAX]])
    keep = rng.permutation(keep)[: int(CAP * human_n[c])]
    picked[c] = keep
    print(f"{CLASSES[c]:<11}{t:>7.2f}{len(idx):>7,}{art.sum():>10,}{nb:>8,}{len(keep):>7,}{human_n[c]:>8,}")

In [ ]:
#@title 7. Look before you train — 24 random pseudo-labels per class vs. real examples
# Count the wrong ones. 0-2 of 24: fine. 3-5: raise that class's bar. 6+: remove it from
# INCLUDE in cell 0 and re-run from cell 6. Confidence will NOT warn you: the artifacts
# this notebook filters were all labelled at 0.90-1.00.
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
CMAP = ListedColormap(["#e9ecef", "#4575b4", "#ca3b37"])    # not tested / pass / fail

for c, keep in picked.items():
    if not len(keep): continue
    pick = rng.choice(keep, min(24, len(keep)), replace=False)
    fig, axes = plt.subplots(5, 6, figsize=(9, 7.6))
    for ax, i in zip(axes[0], rng.choice(np.where(ys == c)[0], 6, replace=False)):
        ax.imshow(np.rint(Xs[i, 0] * 2).astype(int), cmap=CMAP, vmin=0, vmax=2)
        ax.set_title("human label", fontsize=7)
    for ax, i in zip(axes[1:].ravel(), pick):
        ax.imshow(U[i], cmap=CMAP, vmin=0, vmax=2)
        ax.set_title(f"conf {cf_u[i]:.2f}", fontsize=7)
    for ax in axes.ravel(): ax.axis("off")
    fig.suptitle(f"{CLASSES[c]}: top row human-labelled. Below: {len(pick)} of {len(keep)} "
                 f"pseudo-labels that will be added", fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
#@title 8. Self-trained arm — human + pseudo-labels, same recipe and seeds  (~9 min per seed)
PX = np.concatenate([U[k] for k in picked.values()])[:, None].astype(np.float32) / 2.0
PY = np.concatenate([np.full(len(k), c, np.int64) for c, k in picked.items()])
XC, YC = np.concatenate([X_train, PX]), np.concatenate([y_train, PY])
SW = np.concatenate([np.ones(len(y_train)), np.full(len(PY), PSEUDO_W)])
print(f"training on {len(y_train):,} human + {len(PY):,} pseudo-labels (weight {PSEUDO_W})")

for s in SEEDS:
    m = train(XC, YC, SW, s, f"self-trained seed {s}")
    results["self-trained"].append(score(m))
    print(f"self-trained  seed {s}: TEST macro-F1 {results['self-trained'][-1][0]:.4f}")

In [ ]:
#@title 9. Result — the test half, used for the first and only time
print(f"{'':<14}" + "".join(f"seed {s:<7}" for s in SEEDS) + "mean")
for k, rs in results.items():
    print(f"{k:<14}" + "".join(f"{r[0]:<12.4f}" for r in rs) + f"{np.mean([r[0] for r in rs]):.4f}")

delta = [b[0] - a[0] for a, b in zip(results["baseline"], results["self-trained"])]
print(f"\npaired delta per seed: {np.round(delta, 4)}")
spread = np.ptp([r[0] for r in results["baseline"]]) if len(SEEDS) > 1 else None
if spread is not None:
    print(f"baseline seed-to-seed spread: {spread:.4f}")

bp = np.mean([r[1] for r in results["baseline"]], 0)
sp = np.mean([r[1] for r in results["self-trained"]], 0)
print(f"\n{'class':<11}{'test n':>7}{'baseline':>10}{'self-tr.':>10}{'delta':>8}")
for c in range(NC):
    flag = "   (too few to read)" if (yt == c).sum() < 50 else ""
    print(f"{CLASSES[c]:<11}{(yt == c).sum():>7}{bp[c]:>10.3f}{sp[c]:>10.3f}{sp[c]-bp[c]:>+8.3f}{flag}")

if spread is None:
    verdict = "single seed — report as one run, not a result"
elif all(x > 0 for x in delta) and min(delta) > spread:
    verdict = "IMPROVES: gain on every seed, larger than seed-to-seed noise"
elif all(x > 0 for x in delta):
    verdict = "INCONCLUSIVE: positive on every seed, but within seed-to-seed noise"
else:
    verdict = "NO IMPROVEMENT: not positive on every seed"
print(f"\nverdict: {verdict}")

## Reading the result

- **Look at the verdict and the per-class deltas together.** A macro-F1 gain that comes
  from one class while another drops (e.g. Center up, Local down) is a trade-off, not an
  improvement.
- **Ignore per-class rows marked "too few to read".** Near-full has about 11 test wafers,
  so one prediction moves its F1 by about 0.05.
- **Absolute numbers will differ from the shipped 0.9232.** The test set is half the size
  and checkpoints are chosen on the other half. Only the *difference* between arms matters.

Whatever the verdict, the inspection in cells 6-7 is a finding in its own right: the
unlabelled data contains test-artifact patterns outside the nine classes, which the model
labels at near-100% confidence. The fix for that is an explicit "unknown / artifact" class,
not more pseudo-labels.